# Local Offline RAG with Ollama

## Overview
This notebook demonstrates building a **completely offline RAG (Retrieval-Augmented Generation)** system using **Ollama** for local LLMs and embeddings.

### 🚀 Benefits of Local RAG:
- **100% Offline**: No internet required after setup
- **Privacy First**: Your documents never leave your machine
- **No API Costs**: Free to run unlimited queries
- **Fast**: No network latency
- **Full Control**: Customize models and parameters

### 📋 Architecture:
```
PDF Documents → Load → Split → Local Embeddings (Ollama) → ChromaDB
                                                                  ↓
User Query → Retrieve Similar Chunks → Local LLM (Ollama) → Answer
```

### 🛠️ Components:
- **Document Loader**: PyPDFLoader
- **Text Splitter**: RecursiveCharacterTextSplitter
- **Embeddings**: Ollama with nomic-embed-text (or embeddinggemma)
- **Vector Store**: ChromaDB (persistent, local)
- **LLM**: Ollama with gemma3:1b
- **Chain**: LangChain Expression Language (LCEL)

---

## 1. Prerequisites & Installation

### Required Software:
1. **Ollama**: Download from https://ollama.ai
2. **Python 3.9+**: Recommended 3.11 or 3.13

### Install Python Packages:

In [ ]:
# Install required packages (run this once)
# !pip install langchain langchain-core langchain-community langchain-text-splitters
# !pip install langchain-ollama langchain-chroma chromadb
# !pip install pypdf tiktoken

# Or install from requirements.txt with additional packages:
# !pip install -r requirements.txt
# !pip install langchain-ollama langchain-chroma chromadb

### Download Ollama Models:

Run these commands in your terminal (if you haven't already):

```bash
# Embedding model (choose one or both)
ollama pull nomic-embed-text    # Recommended: 274 MB
ollama pull embeddinggemma      # Alternative: 621 MB

# LLM for generation
ollama pull gemma3:1b          # Small & fast: 815 MB
```

**Note**: You already have these models downloaded! ✓

## 2. Import Required Libraries

In [ ]:
# Standard library imports
import os
import sys
from pathlib import Path

# LangChain Document Loaders
from langchain_community.document_loaders import PyPDFLoader

# LangChain Text Splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Ollama Integration
from langchain_ollama import OllamaEmbeddings, ChatOllama

# ChromaDB Vector Store
from langchain_chroma import Chroma

# LangChain Core Components
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("✓ All imports successful!")
print("✓ Ready for local offline RAG!")
print(f"\nPython version: {sys.version}")

## 3. Verify Ollama Installation

Let's check that Ollama is running and our models are available.

In [ ]:
# Check Ollama is running and list available models
!ollama list

In [ ]:
# Test Ollama connection with a simple query
print("Testing Ollama connection...\n")

try:
    test_llm = ChatOllama(model="gemma3:270m", temperature=0)
    response = test_llm.invoke("Say 'Hello! I am running locally on your machine!'")
    
    print("✓ Ollama is working!")
    print(f"Response: {response.content}")
    
except Exception as e:
    print(f"✗ Error connecting to Ollama: {e}")
    print("\nMake sure Ollama is running. Try: ollama serve")

## 4. Load PDF Documents

Load your PDF documents for the RAG system.

In [ ]:
# ===== CONFIGURATION: Update this path to your PDF file =====
pdf_path = "attention.pdf"  # Change this to your PDF file path
# =============================================================

# Check if file exists
if not os.path.exists(pdf_path):
    print(f"⚠️  ERROR: File '{pdf_path}' not found!")
    print("Please update the pdf_path variable with your PDF file location.")
else:
    # Load the PDF
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    
    # Display information
    print(f"✓ Loaded {len(documents)} pages from '{pdf_path}'")
    print(f"\n--- First Page Preview ---")
    print(f"Content (first 300 chars): {documents[0].page_content[:300]}...")
    print(f"\nMetadata: {documents[0].metadata}")
    print(f"\nTotal characters: {sum(len(doc.page_content) for doc in documents):,}")

### Optional: Load Multiple PDFs from a Directory

In [ ]:
# Uncomment to load multiple PDFs from a directory

# pdf_directory = "./pdfs"
# all_documents = []

# if os.path.exists(pdf_directory):
#     pdf_files = list(Path(pdf_directory).glob("*.pdf"))
#     print(f"Found {len(pdf_files)} PDF files\n")
    
#     for pdf_file in pdf_files:
#         loader = PyPDFLoader(str(pdf_file))
#         docs = loader.load()
#         all_documents.extend(docs)
#         print(f"  ✓ Loaded {len(docs)} pages from {pdf_file.name}")
    
#     print(f"\nTotal pages loaded: {len(all_documents)}")
#     documents = all_documents  # Use this for the rest of the pipeline

## 5. Split Documents into Chunks

Break documents into smaller chunks for better retrieval precision.

In [ ]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,        # Characters per chunk
    chunk_overlap=128,      # Overlap to maintain context
    length_function=len,
    separators=["\n\n", "\n", " ", ""]  # Split on paragraphs, then lines, etc.
)

# Split documents
chunks = text_splitter.split_documents(documents)

# Display results
avg_chunk_size = sum(len(chunk.page_content) for chunk in chunks) / len(chunks)

print(f"✓ Split {len(documents)} documents into {len(chunks)} chunks")
print(f"\nAverage chunk size: {avg_chunk_size:.0f} characters")

# Preview chunks
print(f"\n--- Chunk Examples ---")
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1} (length: {len(chunk.page_content)} chars):")
    print(f"{chunk.page_content[:200]}...")

## 6. Create Embeddings (Primary: Nomic-Embed-Text)

### About Nomic-Embed-Text:
- **Size**: 274 MB
- **Dimensions**: 768
- **Performance**: State-of-the-art for local embeddings
- **Speed**: Fast inference
- **License**: Open source (Apache 2.0)

In [ ]:
# Initialize Ollama Embeddings with nomic-embed-text
embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
    # base_url="http://localhost:11434"  # Default Ollama URL
)

# Test embeddings
print("Testing nomic-embed-text embeddings...\n")
sample_text = "This is a test sentence for embeddings."
sample_embedding = embeddings.embed_query(sample_text)

print(f"✓ Embeddings model: nomic-embed-text")
print(f"✓ Embedding dimension: {len(sample_embedding)}")
print(f"✓ Sample embedding (first 10 values): {sample_embedding[:10]}")
print(f"\nℹ️  Each chunk will be converted to a {len(sample_embedding)}-dimensional vector")
print(f"ℹ️  All processing happens locally on your machine!")

## 7. Alternative: EmbeddingGemma (Optional)

### About EmbeddingGemma:
- **Size**: 621 MB (larger than nomic)
- **Dimensions**: 768
- **Optimized for**: Google Gemma models
- **Use case**: Better alignment with Gemma LLMs

**Uncomment the code below to use embeddinggemma instead:**

In [ ]:
# # Alternative: Use embeddinggemma instead
# embeddings = OllamaEmbeddings(
#     model="embeddinggemma:latest"
# )

# # Test embeddings
# print("Testing embeddinggemma embeddings...\n")
# sample_text = "This is a test sentence for embeddings."
# sample_embedding = embeddings.embed_query(sample_text)

# print(f"✓ Embeddings model: embeddinggemma")
# print(f"✓ Embedding dimension: {len(sample_embedding)}")
# print(f"✓ Sample embedding (first 10 values): {sample_embedding[:10]}")

## 8. Create ChromaDB Vector Store

### Why ChromaDB?
- **Local & Persistent**: Stores vectors on disk
- **Python 3.13 Compatible**: Works with latest Python
- **Easy to Use**: Simple API
- **Open Source**: Free and fully featured

**Note**: This step may take a minute as it processes all chunks.

In [ ]:
# Create ChromaDB vector store
print(f"Creating ChromaDB vector store from {len(chunks)} chunks...")
print("This may take a minute...\n")

# Set persistent directory
persist_directory = "./chroma_db"

# Create vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="local_rag_collection"
)

print(f"✓ ChromaDB vector store created successfully!")
print(f"✓ Indexed {len(chunks)} document chunks")
print(f"✓ Stored at: {persist_directory}")
print(f"\nℹ️  Vector store persisted to disk - you can reload it later!")

### Load Existing Vector Store (Optional)

If you've already created the vector store, you can load it instead:

In [ ]:
# # Uncomment to load existing vector store
# persist_directory = "./chroma_db"

# vectorstore = Chroma(
#     persist_directory=persist_directory,
#     embedding_function=embeddings,
#     collection_name="local_rag_collection"
# )

# print(f"✓ Loaded existing vector store from '{persist_directory}'")
# print(f"✓ Collection: local_rag_collection")

## 9. Create Retriever and Test

The retriever finds the most relevant chunks for a given query.

In [ ]:
# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",    # Use cosine similarity
    search_kwargs={"k": 4}        # Retrieve top 4 most relevant chunks
)

print("✓ Retriever configured successfully")
print(f"  - Search type: similarity")
print(f"  - Number of documents to retrieve (k): 4")

# Test retrieval
test_query = "What is the main topic of this document?"
print(f"\n--- Retriever Test ---")
print(f"Query: '{test_query}'")

retrieved_docs = retriever.invoke(test_query)

print(f"\nRetrieved {len(retrieved_docs)} documents:")
for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i+1}:")
    print(f"  Content preview: {doc.page_content[:150]}...")
    print(f"  Source: Page {doc.metadata.get('page', 'N/A')}")

## 10. Configure Ollama LLM (Gemma3:1b)

### About Gemma3:1b:
- **Size**: 815 MB
- **Parameters**: 1 billion
- **Speed**: Very fast inference
- **Quality**: Good for most Q&A tasks
- **Memory**: Low RAM usage

**Alternatives**: You can also use llama3.2 (2GB) or deepseek-r1 (4.7GB) for better quality.

In [ ]:
# Initialize Ollama LLM
llm = ChatOllama(
    model="gemma3:270m",
    temperature=0,          # Deterministic responses (0 = focused, 1 = creative)
    # num_predict=2000,     # Max tokens to generate
    # top_k=40,             # Top-k sampling
    # top_p=0.9,            # Top-p (nucleus) sampling
)

print("✓ LLM configured successfully")
print(f"  - Model: gemma3:1b (local)")
print(f"  - Temperature: 0 (deterministic)")

# Test LLM
test_response = llm.invoke("Say 'Hello! I am Gemma running locally!'")
print(f"\nLLM Test Response: {test_response.content}")

### Try Other Local Models (Optional)

In [ ]:
# # Alternative: Use llama3.2 for better quality
# llm = ChatOllama(
#     model="llama3.2:latest",
#     temperature=0
# )

# # Alternative: Use deepseek-r1 for reasoning tasks
# llm = ChatOllama(
#     model="deepseek-r1:latest",
#     temperature=0
# )

## 11. Build RAG Chain (LangChain Expression Language)

Combine retrieval and generation into a single pipeline using LCEL.

In [ ]:
# Define prompt template
system_prompt = (
    "You are a helpful assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer based on the context, say that you don't know. "
    "Keep the answer concise and accurate.\n\n"
    "Context: {context}\n\n"
    "Question: {question}"
)

prompt = ChatPromptTemplate.from_template(system_prompt)

# Helper function to format documents
def format_docs(docs):
    """Format retrieved documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

# Build RAG chain using LCEL
rag_chain = (
    {
        "context": retriever | format_docs,  # Retrieve and format docs
        "question": RunnablePassthrough()      # Pass through the question
    }
    | prompt           # Format with prompt template
    | llm              # Generate answer with local LLM
    | StrOutputParser() # Parse output to string
)

print("✓ RAG chain created successfully using LCEL!")
print("\nRAG Pipeline Flow:")
print("  1. User provides a query")
print("  2. Retriever finds top 4 relevant chunks (local ChromaDB)")
print("  3. Chunks are formatted as context")
print("  4. Context + question formatted with prompt template")
print("  5. Local LLM (gemma3:1b) generates answer")
print("  6. Answer parsed and returned")
print("\n🔒 Everything runs locally on your machine!")

## 12. Test RAG Pipeline with Example Queries

Let's test our complete local RAG system!

In [ ]:
# Example Query 1: General question
query1 = "What is summary of this document?"

print(f"Query: {query1}")
print("\nProcessing locally...\n")

answer = rag_chain.invoke(query1)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(answer)
print("\n" + "=" * 80)

# Show source documents
print("\nSOURCE DOCUMENTS USED:")
print("=" * 80)
retrieved_docs = retriever.invoke(query1)
for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i+1}:")
    print(f"  Page: {doc.metadata.get('page', 'N/A')}")
    print(f"  Content: {doc.page_content[:200]}...")
    print("-" * 80)

In [ ]:
# Example Query 2: Specific information extraction
query2 = "Can you summarize the key technical contributions or innovations mentioned?"

print(f"Query: {query2}")
print("\nProcessing locally...\n")

answer = rag_chain.invoke(query2)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(answer)
print("\n" + "=" * 80)

In [ ]:
# Example Query 3: Your custom question
custom_query = "What specific details are mentioned about the methodology or approach?"

print(f"Query: {custom_query}")
print("\nProcessing locally...\n")

answer = rag_chain.invoke(custom_query)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(answer)
print("\n" + "=" * 80)

## 13. Interactive Q&A Session

Ask your own questions to the RAG system!

In [ ]:
# Interactive Q&A
def ask_question(question):
    """Ask a question to the RAG system."""
    print(f"\n{'='*80}")
    print(f"Question: {question}")
    print(f"{'='*80}")
    
    answer = rag_chain.invoke(question)
    
    print(f"\nAnswer: {answer}")
    print(f"{'='*80}\n")
    
    return answer

# Try it out!
# Change the question below to ask anything about your document
my_question = "What are the main findings or results?"
ask_question(my_question)

## 14. Bonus: Compare Embedding Models (Optional)

Compare retrieval results between nomic-embed-text and embeddinggemma.

In [ ]:
# # Uncomment to compare embedding models

# print("Comparing embedding models...\n")

# test_query = "What is attention mechanism?"

# # Test with nomic-embed-text
# print("=" * 80)
# print("Using nomic-embed-text:")
# print("=" * 80)
# embeddings_nomic = OllamaEmbeddings(model="nomic-embed-text")
# vectorstore_nomic = Chroma.from_documents(
#     documents=chunks[:10],  # Use first 10 chunks for quick test
#     embedding=embeddings_nomic,
#     collection_name="test_nomic"
# )
# retriever_nomic = vectorstore_nomic.as_retriever(search_kwargs={"k": 2})
# docs_nomic = retriever_nomic.invoke(test_query)

# print(f"\nTop retrieved document:")
# print(f"{docs_nomic[0].page_content[:200]}...\n")

# # Test with embeddinggemma
# print("=" * 80)
# print("Using embeddinggemma:")
# print("=" * 80)
# embeddings_gemma = OllamaEmbeddings(model="embeddinggemma:latest")
# vectorstore_gemma = Chroma.from_documents(
#     documents=chunks[:10],
#     embedding=embeddings_gemma,
#     collection_name="test_gemma"
# )
# retriever_gemma = vectorstore_gemma.as_retriever(search_kwargs={"k": 2})
# docs_gemma = retriever_gemma.invoke(test_query)

# print(f"\nTop retrieved document:")
# print(f"{docs_gemma[0].page_content[:200]}...\n")

# print("=" * 80)
# print("\nℹ️  Both models perform well. Choose based on your preference!")
# print("   - nomic-embed-text: Smaller (274MB), general-purpose")
# print("   - embeddinggemma: Larger (621MB), optimized for Gemma models")

## 15. Performance Tips & Next Steps

### 🚀 Performance Optimization:
1. **Chunk Size**: Experiment with different sizes (512, 1024, 2048)
2. **Retrieval Count (k)**: Try k=3, 4, 5, 6 based on your needs
3. **Model Selection**: 
   - Fast: gemma3:1b
   - Balanced: llama3.2
   - Best quality: deepseek-r1
4. **Temperature**: 0 for factual, 0.3-0.7 for creative

### 💾 Persistence:
- Vector store is saved at `./chroma_db`
- You can reload it without re-embedding documents
- Delete the directory to start fresh

### 🔧 Troubleshooting:
- **Slow responses**: Use smaller model (gemma3:1b)
- **Out of memory**: Reduce chunk count or use smaller model
- **Ollama not found**: Make sure `ollama serve` is running

### 📚 Next Steps:
1. Try different documents and PDFs
2. Experiment with other Ollama models
3. Add custom preprocessing or post-processing
4. Build a simple UI with Gradio or Streamlit
5. Compare with cloud-based RAG (OpenAI, etc.)

### 🎉 Congratulations!
You now have a fully functional **local, offline RAG system** running on your machine!

---

**Created with LangChain + Ollama + ChromaDB**